# 🗜️ Tool nén PDF trong Google Drive

Nén toàn bộ file PDF trong thư mục **Lần 2** → xuất file nén **cùng tên** vào thư mục **Lần 2 NÉN**.

**Cách dùng:** Chạy lần lượt 3 cell bên dưới (Cell 1 → Cell 2 → Cell 3).

In [ ]:
# ============ CELL 1: Kết nối Google Drive ============
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============ CELL 2: Cài Ghostscript (công cụ nén PDF) ============
!apt-get -qq update && apt-get -qq install -y ghostscript > /dev/null
!gs --version

In [ ]:
# ============ CELL 3: Nén toàn bộ PDF ============
import os, shutil, subprocess
from pathlib import Path

# --- CẤU HÌNH ---
SRC = Path('/content/drive/MyDrive/MSC - 2026/Travel to Ireland/Chuyển tiền học phí/Chứng Minh Tài Chính/Lần 2')
DST = Path('/content/drive/MyDrive/MSC - 2026/Travel to Ireland/Chuyển tiền học phí/Chứng Minh Tài Chính/Lần 2 NÉN')

# Mức nén: '/screen' = nén mạnh nhất (72 dpi), '/ebook' = cân bằng (150 dpi, khuyên dùng),
#          '/printer' = chất lượng cao (300 dpi, nén ít)
QUALITY = '/ebook'
# -----------------

assert SRC.exists(), f'❌ Không tìm thấy thư mục gốc: {SRC}'
DST.mkdir(parents=True, exist_ok=True)

pdfs = sorted(SRC.glob('*.pdf'))
print(f'Tìm thấy {len(pdfs)} file PDF trong thư mục gốc.\n')

total_in, total_out = 0, 0
for i, pdf in enumerate(pdfs, 1):
    out = DST / pdf.name
    tmp = Path('/content') / f'tmp_{i}.pdf'

    cmd = [
        'gs', '-sDEVICE=pdfwrite',
        '-dCompatibilityLevel=1.4',
        f'-dPDFSETTINGS={QUALITY}',
        '-dNOPAUSE', '-dQUIET', '-dBATCH',
        '-dDetectDuplicateImages=true',
        '-dCompressFonts=true',
        f'-sOutputFile={tmp}',
        str(pdf),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    size_in = pdf.stat().st_size
    if result.returncode != 0 or not tmp.exists() or tmp.stat().st_size == 0:
        # Nén lỗi → copy nguyên bản file gốc
        shutil.copy2(pdf, out)
        size_out = size_in
        note = '⚠️ nén lỗi, giữ file gốc'
    elif tmp.stat().st_size >= size_in:
        # File nén to hơn file gốc → giữ file gốc
        shutil.copy2(pdf, out)
        size_out = size_in
        note = '↔️ file gốc đã tối ưu, giữ nguyên'
    else:
        shutil.copy2(tmp, out)
        size_out = out.stat().st_size
        note = f'✅ giảm {(1 - size_out/size_in)*100:.0f}%'
    tmp.unlink(missing_ok=True)

    total_in += size_in
    total_out += size_out
    print(f'[{i:2d}/{len(pdfs)}] {pdf.name}')
    print(f'         {size_in/1e6:6.2f} MB → {size_out/1e6:6.2f} MB   {note}\n')

print('=' * 60)
print(f'TỔNG: {total_in/1e6:.1f} MB → {total_out/1e6:.1f} MB '
      f'(giảm {(1 - total_out/total_in)*100:.0f}%)')
print(f'📁 File nén nằm tại: {DST}')

---
**Mẹo:** Nếu file nén vẫn còn lớn, đổi `QUALITY = '/ebook'` thành `'/screen'` ở Cell 3 rồi chạy lại (nén mạnh hơn nhưng ảnh scan sẽ mờ hơn). Với hồ sơ chứng minh tài chính cần chữ rõ nét, `/ebook` (150 dpi) là lựa chọn an toàn.